# v5 — auditoria multi-corpus

A afirmacao central do artigo — *kernels de fidelidade acompanham surrogates classicos* — e' sobre
o **metodo**, nao sobre voz. Demonstra-la num unico corpus e' a primeira objecao que um revisor
metodologico levanta. Aqui o mesmo protocolo roda sobre varios conjuntos tabulares.

**O dataset que vale mais que os outros: `ad_hoc` (Havlicek et al., 2019).** E' sintetico e foi
construido de proposito para ser separavel por um kernel quantico com feature map ZZ — o seu.
Funciona como **controle positivo**: se o protocolo detecta a separacao onde ela existe por
construcao, ele esta calibrado, e o resultado nulo nos dados reais significa ausencia de efeito.
Se **nao** detectar, o protocolo e' cego — e isso seria o achado mais importante de todos, porque
invalidaria as conclusoes anteriores.

Rode o `ad_hoc` primeiro. Se ele falhar, pare e reveja o protocolo antes de qualquer outra coisa.

**Dois bracos apenas** (`fid` e `proj`). O braco `fid-fixo` da v4 ja respondeu a pergunta de
desconfundimento e nao precisa ser refeito por dataset.

In [ ]:
# !pip install pennylane -q
import numpy as np, pandas as pd, time, os, io, urllib.request
import pennylane as qml
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score
from scipy.optimize import minimize_scalar
from scipy.stats import spearmanr, t as tdist
from IPython.display import HTML, display
import warnings; warnings.filterwarnings("ignore")

BG, FG, OR = "#0d0d0d", "#e8e8e8", "#ff6b00"
FONT = "'JetBrains Mono','Fira Code',monospace"

# Painel escuro com acento laranja no lugar de print cru.
def painel(titulo, linhas, tabela=None, destaque=None):
    corpo = "".join(f"<div style='margin:.28em 0'><span style='color:{OR}'>$</span> {l}</div>"
                    for l in linhas)
    tab = ""
    if tabela is not None:
        tab = (tabela.style.hide(axis="index").set_table_styles([
            {"selector": "th", "props": [("color", OR), ("text-align", "right"),
                                         ("border-bottom", f"1px solid {OR}"), ("padding", "4px 12px")]},
            {"selector": "td", "props": [("text-align", "right"), ("padding", "3px 12px"),
                                         ("border-bottom", "1px solid #222")]}])
            .set_table_attributes(f"style='border-collapse:collapse;font-family:{FONT};"
                                  f"font-size:12px;color:{FG};margin:.8em 0 .2em 0'").to_html())
    dest = (f"<div style='margin-top:.9em;padding:.6em .8em;border-left:3px solid {OR};"
            f"background:#1a1206;color:{FG}'>{destaque}</div>") if destaque else ""
    display(HTML(f"<div style='background:{BG};color:{FG};font-family:{FONT};font-size:13px;"
                 f"padding:1.1em 1.3em;border:1px solid #222;border-radius:6px'>"
                 f"<div style='color:{OR};font-size:11px;letter-spacing:.16em;margin-bottom:.7em'>"
                 f"&#9632; {titulo.upper()}</div>{corpo}{tab}{dest}</div>"))

## 1. Carregadores

Cada carregador devolve `(X, y, grupos, nome)`. Quando nao ha estrutura de sujeito, `grupos` e'
o proprio indice — o `StratifiedGroupKFold` degenera para `StratifiedKFold`, que e' o correto
nesse caso.

O `wdbc.data` precisa estar no diretorio (faca upload pelo painel do Colab). Os demais sao
gerados ou baixados.

In [ ]:
def carrega_adhoc(n=400, dim=3, gap=0.3, seed=0):
    """Havlicek et al. 2019: rotulo dado por um observavel no espaco do feature map ZZ.
    Separavel por construcao para o kernel quantico — controle positivo."""
    rng = np.random.default_rng(seed)
    dev = qml.device("default.qubit", wires=dim)
    @qml.qnode(dev)
    def phi(x):
        for _ in range(2):
            for i in range(dim):
                qml.Hadamard(wires=i); qml.RZ(2.0 * x[..., i], wires=i)
            for i in range(dim - 1):
                qml.CNOT(wires=[i, i + 1])
                qml.RZ(2.0 * (np.pi - x[..., i]) * (np.pi - x[..., i + 1]), wires=i + 1)
                qml.CNOT(wires=[i, i + 1])
        return qml.state()
    V = np.linalg.qr(rng.normal(size=(2**dim, 2**dim)) +
                     1j * rng.normal(size=(2**dim, 2**dim)))[0]
    D = np.diag([1.0] * (2**(dim-1)) + [-1.0] * (2**(dim-1)))
    O = V.conj().T @ D @ V                       # observavel aleatorio
    X, y = [], []
    while len(X) < n:
        cand = rng.uniform(0, 2 * np.pi, (2000, dim))
        S = np.asarray(phi(cand))
        val = np.einsum('ni,ij,nj->n', S.conj(), O, S).real
        keep = np.abs(val) > gap
        for xx, vv in zip(cand[keep], val[keep]):
            X.append(xx); y.append(int(vv > 0))
            if len(X) >= n: break
    X, y = np.array(X), np.array(y)
    return X, y, np.arange(len(y)), "ad_hoc"

def carrega_wdbc(path="wdbc.data"):
    cols = ["id", "diag"] + [f"f{i}" for i in range(30)]
    d = pd.read_csv(path, header=None, names=cols)
    return (d[[f"f{i}" for i in range(30)]].values.astype(float),
            (d.diag == "M").astype(int).values, np.arange(len(d)), "wdbc")

def carrega_svd(path="features_v5.pkl"):
    META = ["speaker_id", "session_id", "label", "diagnosis", "vowel", "pitch", "wav"]
    f = pd.read_pickle(path)
    F = [x for x in f.columns if x not in META]
    return (f[F].values.astype(float), f.label.values.astype(int),
            f.speaker_id.values, "svd")

def carrega_uci(url, nome, alvo_pos, sep=",", header=None, drop=None):
    txt = urllib.request.urlopen(url).read().decode()
    d = pd.read_csv(io.StringIO(txt), sep=sep, header=header)
    if drop: d = d.drop(columns=drop)
    y = (d.iloc[:, -1].astype(str) == alvo_pos).astype(int).values
    X = d.iloc[:, :-1].apply(pd.to_numeric, errors="coerce").values.astype(float)
    return X, y, np.arange(len(y)), nome

## 2. Protocolo

Identico ao da v4: subamostragem por grupo, imputacao e escala ajustadas so no treino,
selecao das `k` melhores features, bandwidth escolhida por leave-one-fold-out na analise.

O unico ajuste e' `N_PER_CLASS`: datasets pequenos (wdbc tem 212 malignos) nao suportam 200 por
classe em 10 folds. O carregador reporta o teto e o codigo usa o menor entre o pedido e o
disponivel.

In [ ]:
def _state_fn(q, reps, ent):
    dev = qml.device("default.qubit", wires=q)
    @qml.qnode(dev)
    def circ(x):
        for _ in range(reps):
            for i in range(q):
                qml.Hadamard(wires=i); qml.RZ(2.0 * x[..., i], wires=i)
            if ent:
                for i in range(q - 1):
                    qml.CNOT(wires=[i, i + 1])
                    qml.RZ(2.0 * (np.pi - x[..., i]) * (np.pi - x[..., i + 1]), wires=i + 1)
                    qml.CNOT(wires=[i, i + 1])
        return qml.state()
    return circ

def states(X, q, reps=2, ent=True, batch=512):
    f = _state_fn(q, reps, ent)
    return np.vstack([np.asarray(f(X[s:s + batch])) for s in range(0, len(X), batch)])

def gram_fid(A, B): return np.abs(A @ B.conj().T) ** 2

def bloch(S, q):
    n = S.shape[0]; psi = S.reshape((n,) + (2,) * q); out = np.empty((n, 3 * q))
    for k in range(q):
        m = np.moveaxis(psi, 1 + k, 1).reshape(n, 2, -1)
        rho = np.einsum('nik,njk->nij', m, m.conj())
        out[:, 3*k] = 2*rho[:, 0, 1].real
        out[:, 3*k+1] = -2*rho[:, 0, 1].imag
        out[:, 3*k+2] = (rho[:, 0, 0]-rho[:, 1, 1]).real
    return out

def gram_proj(Ba, Bb, g): return np.exp(-g * ((Ba[:, None, :]-Bb[None, :, :])**2).sum(-1))

def mediana_gamma(Ba):
    D2 = ((Ba[:, None, :]-Ba[None, :, :])**2).sum(-1)
    m = np.median(D2[np.triu_indices_from(D2, k=1)])
    return 1.0/m if m > 0 else 1.0

def fit_surrogate(K, Xa):
    D2 = ((Xa[:, None, :]-Xa[None, :, :])**2).sum(-1)
    r = minimize_scalar(lambda lg: np.linalg.norm(K - np.exp(-np.exp(lg)*D2)),
                        bounds=(-8, 6), method="bounded")
    return float(np.exp(r.x)), float(r.fun)/np.sqrt(K.size)

def off_diag_sd(K): return float(K[np.triu_indices_from(K, k=1)].std())

def subsample(ytr, gtr, npc, rng):
    idx = []
    for cls in (0, 1):
        take = []
        for s in rng.permutation(np.unique(gtr[ytr == cls])):
            take += list(np.where((gtr == s) & (ytr == cls))[0])
            if len(take) >= npc: break
        idx += take[:npc]
    return np.array(idx)

In [ ]:
QUBITS = [4, 6, 8, 10]
SCALES = np.round(np.geomspace(0.01, 2.0, 12), 4)
MAPAS = [("zz", True), ("prod", False)]
BRACOS = [("fid", "fid"), ("proj", "proj")]
N_PER_CLASS, N_FOLDS, N_REPS = 200, 10, 1
C_SVM, REPS_CIRCUITO, MARGEM_EQUIV = 1.0, 2, 0.01

def roda(loader, npc=N_PER_CLASS):
    X, y, g, nome = loader()
    npc = min(npc, int(np.bincount(y).min() * (N_FOLDS - 1) / N_FOLDS))
    qs = [q for q in QUBITS if q <= X.shape[1]]
    rows = []
    sg = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=7)
    t0 = time.time()
    for fold, (tr, te) in enumerate(sg.split(X, y, g)):
        for rep in range(N_REPS):
            rng = np.random.default_rng(1000*fold + rep)
            sub = subsample(y[tr], g[tr], npc, rng)
            ytr, yte = y[tr][sub], y[te]
            imp = SimpleImputer(strategy="median").fit(X[tr][sub])
            Xtr, Xte = imp.transform(X[tr][sub]), imp.transform(X[te])
            for q in qs:
                sel = SelectKBest(f_classif, k=q).fit(Xtr, ytr)
                sc = MinMaxScaler((0, np.pi)).fit(sel.transform(Xtr))
                A = sc.transform(sel.transform(Xtr))
                B = np.clip(sc.transform(sel.transform(Xte)), 0, np.pi)
                for cv in SCALES:
                    Ac, Bc = A*cv, B*cv
                    for mp, ent in MAPAS:
                        S1, S2 = states(Ac, q, REPS_CIRCUITO, ent), states(Bc, q, REPS_CIRCUITO, ent)
                        for braco, kern in BRACOS:
                            if kern == "fid":
                                K1, K2 = gram_fid(S1, S1), gram_fid(S2, S1)
                            else:
                                P1, P2 = bloch(S1, q), bloch(S2, q)
                                gp = mediana_gamma(P1)
                                K1, K2 = gram_proj(P1, P1, gp), gram_proj(P2, P1, gp)
                            m = SVC(kernel="precomputed", C=C_SVM).fit(K1, ytr)
                            gs, resid = fit_surrogate(K1, Ac)
                            ms = SVC(kernel="rbf", gamma=gs, C=C_SVM).fit(Ac, ytr)
                            rows.append(dict(dataset=nome, braco=braco, mapa=mp, fold=fold,
                                             rep=rep, n_qubits=q, c=cv,
                                             auc_q=roc_auc_score(yte, m.decision_function(K2)),
                                             auc_surr=roc_auc_score(yte, ms.decision_function(Bc)),
                                             resid=resid, gram_off_sd=off_diag_sd(K1)))
    d = pd.DataFrame(rows)
    d.to_csv(f"v5_{nome}.csv", index=False)
    painel(f"{nome} concluido",
           [f"{X.shape[0]} amostras &middot; {X.shape[1]} features &middot; "
            f"{pd.unique(g).size} grupos &middot; prevalencia={y.mean():.3f}",
            f"{npc} por classe no treino &middot; qubits testados: {qs}",
            f"{len(d)} linhas em {time.time()-t0:.0f}s &rarr; v5_{nome}.csv"],
           tabela=d.groupby("braco").auc_q.agg(["max", "mean"]).round(4).reset_index())
    return d

## 3. Controle positivo — rode esta celula primeiro

Se o kernel de fidelidade **nao** superar o surrogate classico aqui, o protocolo nao consegue
detectar vantagem quantica nem onde ela existe por construcao. Nesse caso pare: as conclusoes
das rodadas anteriores nao se sustentam, e o problema esta no protocolo, nao nos dados.

In [ ]:
d_adhoc = roda(carrega_adhoc)

g = d_adhoc[d_adhoc.braco == "fid"]
a = g.groupby(["mapa", "n_qubits", "c"])[["auc_q", "auc_surr"]].mean().reset_index()
zz = a[a.mapa == "zz"]
i = zz.auc_q.idxmax()
melhor = zz.loc[i]
r, p = spearmanr(zz.auc_q, zz.auc_surr)
vantagem = melhor.auc_q - melhor.auc_surr

if vantagem > 0.05:
    dest = (f"<b>PROTOCOLO CALIBRADO</b>: no melhor ponto o kernel quantico supera o surrogate "
            f"em {vantagem:+.4f} de AUC. O protocolo detecta vantagem onde ela existe — "
            "o resultado nulo nos dados reais e' ausencia de efeito, nao cegueira do metodo.")
else:
    dest = (f"<b>PARE</b>: no dataset construido para favorecer o kernel ZZ, a vantagem sobre "
            f"o surrogate e' de apenas {vantagem:+.4f}. Antes de escrever qualquer coisa, "
            "investigue: o protocolo pode nao ser capaz de detectar vantagem alguma.")

painel("controle positivo (ad hoc)",
       [f"melhor celula ZZ: q={int(melhor.n_qubits)}, c={melhor.c}",
        f"AUC quantico = <b>{melhor.auc_q:.4f}</b> &nbsp;|&nbsp; surrogate = {melhor.auc_surr:.4f}",
        f"vantagem no otimo = <b>{vantagem:+.4f}</b>",
        f"Spearman(auc_q, auc_surr) = {r:+.3f} (p={p:.2g})"],
       destaque=dest)

## 4. Corpora reais

In [ ]:
d_wdbc = roda(carrega_wdbc)
# d_svd = roda(carrega_svd)     # descomente se o pickle estiver no diretorio

## 5. Sintese entre datasets

A tabela final e' o que vai para o artigo: uma linha por dataset e braco, com a correlacao
quantico-surrogate e a vantagem no otimo. O padrao entre corpora e' o argumento — nao o valor
absoluto de nenhum deles.

In [ ]:
partes = [x for x in [globals().get("d_adhoc"), globals().get("d_wdbc"),
                      globals().get("d_svd")] if x is not None]
todos = pd.concat(partes)

lin = []
for (ds, br), g in todos.groupby(["dataset", "braco"]):
    a = g[g.mapa == "zz"].groupby(["n_qubits", "c"])[["auc_q", "auc_surr"]].mean().reset_index()
    r, p = spearmanr(a.auc_q, a.auc_surr)
    m = a.loc[a.auc_q.idxmax()]
    lin.append(dict(dataset=ds, braco=br, auc_max=round(m.auc_q, 4),
                    surr_no_otimo=round(m.auc_surr, 4),
                    vantagem=round(m.auc_q - m.auc_surr, 4),
                    rho=round(r, 3), p=f"{p:.1g}"))
sint = pd.DataFrame(lin)

reais = sint[sint.dataset != "ad_hoc"]
ctrl = sint[(sint.dataset == "ad_hoc") & (sint.braco == "fid")]
if len(ctrl) and ctrl.vantagem.iloc[0] > 0.05 and (reais.vantagem.abs() < 0.02).all():
    dest = ("O protocolo detecta vantagem no controle positivo e <b>nao</b> a encontra em "
            "nenhum corpus real. Esse contraste e' o argumento central do artigo.")
elif (reais.vantagem > 0.02).any():
    q = reais[reais.vantagem > 0.02]
    dest = (f"Vantagem acima de 0.02 em: {list(zip(q.dataset, q.braco))}. "
            "Investigue antes de concluir ausencia de efeito.")
else:
    dest = "Reporte a tabela como esta; o padrao entre corpora e' o resultado."

painel("sintese multi-corpus",
       [f"{len(partes)} datasets &middot; {len(todos)} avaliacoes de kernel",
        "vantagem = AUC quantico &minus; AUC surrogate, no otimo de bandwidth"],
       tabela=sint, destaque=dest)

## 6. Notas

- `N_REPS=1` aqui: as repeticoes reamostram dentro dos mesmos folds e nao adicionam informacao
  independente — o que sustenta os intervalos e' `N_FOLDS`. Ajuste se preferir.
- `wdbc` e `ad_hoc` nao tem estrutura de sujeito; `grupos` e' o indice e o split degenera para
  estratificado simples, que e' o correto nesses casos.
- Para acrescentar outro corpus, escreva um carregador que devolva `(X, y, grupos, nome)` e
  chame `roda(seu_carregador)`. O resto da analise nao muda.